# 11.25 — Reward Shaping

Reward shaping adds extra feedback to a reinforcement-learning problem so an agent can learn from progress before the final payoff arrives. In this lesson, we build a tiny gridworld from scratch, shape rewards with a potential function, and verify the key promise: potential-based shaping can speed learning while preserving the optimal policy.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build reward shaping one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math, including the potential-based shaping term and the policy-invariance argument, is made visible. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, tabular values, and small numerical checks.
import matplotlib.pyplot as plt  # heatmaps and learning curves.
np.random.seed(0)  # reproducibility for random exploration.

### 1. Sparse rewards make credit assignment slow

A reward-shaping problem starts with a delayed-reward environment. The agent moves on a grid, receives a cost for each step, and only gets a large payoff at the goal. The difficulty is not that the goal is mysterious; it is that most transitions look almost identical until the agent happens to arrive.

In [ ]:
H_w, W_w = 4, 4  # grid height and width.
goal_w = (3, 3)  # terminal goal state.
start_w = (0, 0)  # start state.
actions_w = np.array([[-1, 0], [1, 0], [0, -1], [0, 1]])  # up, down, left, right.

print("start:", start_w, "goal:", goal_w, "actions:", actions_w.tolist())

▶ What you'll see: a 4×4 world with four deterministic moves.

In [ ]:
def step_w(state_w, action_idx_w):
    if state_w == goal_w:
        return state_w, 0.0, True
    move_w = actions_w[action_idx_w]
    nxt_w = (int(np.clip(state_w[0] + move_w[0], 0, H_w - 1)), int(np.clip(state_w[1] + move_w[1], 0, W_w - 1)))
    done_w = nxt_w == goal_w
    reward_w = 10.0 if done_w else -1.0
    return nxt_w, reward_w, done_w

path_w = [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (3, 3)]
rewards_w = [10.0 if path_w[i + 1] == goal_w else -1.0 for i in range(len(path_w) - 1)]

print("example path length:", len(rewards_w), "rewards:", rewards_w, "return:", sum(rewards_w))

assert sum(rewards_w) == 5.0

▶ What you'll see: five small penalties before the terminal +10, so useful feedback is mostly delayed.

In [ ]:
grid_w = np.zeros((H_w, W_w))
grid_w[start_w] = 1
grid_w[goal_w] = 2
plt.figure(figsize=(3.6, 3.2))
plt.imshow(grid_w, cmap="viridis")
plt.title("1: sparse-reward gridworld")
plt.xticks(range(W_w)); plt.yticks(range(H_w)); plt.colorbar(label="0 empty, 1 start, 2 goal")
plt.show()

▶ What you'll see: the start and goal are far apart, so random exploration rarely sees the terminal payoff early.

*Why it's done this way:* RL optimizes return, not immediate reward. A sparse terminal reward is mathematically valid, but it has high credit-assignment variance: many early actions receive nearly the same local signal even though some move toward states that make the future payoff easier to reach.

### 2. A potential function turns progress into a number

A potential function $\Phi(s)$ assigns each state a scalar score. We choose larger values near the goal, using negative Manhattan distance. This is not the task reward; it is a heuristic measuring progress. The shaping term will compare potential before and after a move.

In [ ]:
def manhattan_w(state_w, target_w=goal_w):
    return abs(state_w[0] - target_w[0]) + abs(state_w[1] - target_w[1])

Phi_w = np.zeros((H_w, W_w))
for r_w in range(H_w):
    for c_w in range(W_w):
        Phi_w[r_w, c_w] = -manhattan_w((r_w, c_w))

print("Phi(start):", Phi_w[start_w], "Phi(goal):", Phi_w[goal_w])

assert Phi_w[start_w] == -6 and Phi_w[goal_w] == 0

▶ What you'll see: the start has potential -6 and the goal has potential 0, so potential increases along shortest paths.

In [ ]:
plt.figure(figsize=(3.8, 3.2))
plt.imshow(Phi_w, cmap="plasma")
plt.colorbar(label="potential Φ(s)")
plt.title("2: potential = -Manhattan distance")
for r_w in range(H_w):
    for c_w in range(W_w):
        plt.text(c_w, r_w, int(Phi_w[r_w, c_w]), ha="center", va="center", color="white")
plt.show()

▶ What you'll see: numbers climb toward 0 as states get closer to the goal.

*Why it's done this way:* A potential is useful because it is state-based, not action-based. It can say “this next state is closer” without directly hard-coding “always move right.” That distinction matters because policy invariance will rely on shaping being a difference of potentials, not an arbitrary action bonus.

### 3. Potential-based shaping rewards changes in potential

Potential-based shaping adds

$$F(s,a,s')=\gamma\Phi(s')-\Phi(s).$$

The shaped reward is $R'(s,a,s')=R(s,a,s')+F(s,a,s')$. If a transition moves to a higher-potential state, $F$ is positive; if it moves away, $F$ is negative.

In [ ]:
gamma_w = 0.9
s_w = (0, 0)
right_next_w, base_right_w, _ = step_w(s_w, 3)
up_next_w, base_up_w, _ = step_w(s_w, 0)
F_right_w = gamma_w * Phi_w[right_next_w] - Phi_w[s_w]
F_up_w = gamma_w * Phi_w[up_next_w] - Phi_w[s_w]

print("right: next", right_next_w, "base", base_right_w, "F", round(F_right_w, 3), "shaped", round(base_right_w + F_right_w, 3))
print("up-wall: next", up_next_w, "base", base_up_w, "F", round(F_up_w, 3), "shaped", round(base_up_w + F_up_w, 3))

assert round(F_right_w, 3) == 1.5 and round(F_up_w, 3) == 0.6

▶ What you'll see: moving right from the start receives more shaped feedback than bumping into the wall.

In [ ]:
F_right_map_w = np.zeros((H_w, W_w))
for r_w in range(H_w):
    for c_w in range(W_w):
        ns_w, _, _ = step_w((r_w, c_w), 3)
        F_right_map_w[r_w, c_w] = gamma_w * Phi_w[ns_w] - Phi_w[r_w, c_w]
plt.figure(figsize=(3.8, 3.2))
plt.imshow(F_right_map_w, cmap="coolwarm")
plt.colorbar(label="F for action right")
plt.title("3: shaping term for moving right")
plt.show()

▶ What you'll see: moving right is helpful on most rows until the boundary, where the bonus weakens.

*Why it's done this way:* The discount appears inside $\gamma\Phi(s')-\Phi(s)$ because shaped rewards will be accumulated with the same discount as environment rewards. That alignment makes the added terms telescope across time, which is exactly what preserves the optimal policy.

### 4. The telescoping sum explains policy invariance

For a trajectory $s_0,s_1,\dots$, the discounted shaping sum is

$$\sum_{t\ge0}\gamma^t(\gamma\Phi(s_{t+1})-\Phi(s_t))=-\Phi(s_0)+\lim_T\gamma^{T+1}\Phi(s_{T+1}).$$

With bounded potentials and $\gamma<1$, the tail vanishes. Starting from the same $s_0$, every policy receives the same additive constant $-\Phi(s_0)$, so action preferences are unchanged.

In [ ]:
path_short_w = [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (3, 3)]
path_detour_w = [(0, 0), (1, 0), (0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (3, 3)]
def discounted_return_w(path_w, shaped_w=False):
    total_w = 0.0
    for t_w in range(len(path_w) - 1):
        r_base_w = 10.0 if path_w[t_w + 1] == goal_w else -1.0
        bonus_w = gamma_w * Phi_w[path_w[t_w + 1]] - Phi_w[path_w[t_w]] if shaped_w else 0.0
        total_w += (gamma_w ** t_w) * (r_base_w + bonus_w)
    return float(total_w)

base_short_w = discounted_return_w(path_short_w, False)
shape_short_w = discounted_return_w(path_short_w, True)
base_detour_w = discounted_return_w(path_detour_w, False)
shape_detour_w = discounted_return_w(path_detour_w, True)

print("base returns:", round(base_short_w, 3), round(base_detour_w, 3))
print("shaped returns:", round(shape_short_w, 3), round(shape_detour_w, 3))

assert base_short_w > base_detour_w and shape_short_w > shape_detour_w

▶ What you'll see: the shorter path is better before and after shaping.

In [ ]:
diff_short_w = shape_short_w - base_short_w
diff_detour_w = shape_detour_w - base_detour_w

print("shaping additions:", round(diff_short_w, 3), round(diff_detour_w, 3))
print("expected constant near -Phi(start):", -Phi_w[start_w])

assert abs(diff_short_w - diff_detour_w) < 2.0
plt.figure(figsize=(4.4, 3))
plt.bar(["short base", "detour base", "short shaped", "detour shaped"], [base_short_w, base_detour_w, shape_short_w, shape_detour_w], color=["gray", "gray", "teal", "teal"])
plt.xticks(rotation=20); plt.ylabel("discounted return"); plt.title("4: shaping keeps the ranking")
plt.show()

▶ What you'll see: shaping raises returns but does not flip which path is preferred.

*Why it's done this way:* Policy invariance is not magic; it is cancellation. The added reward is the discounted difference of a state scalar, so most intermediate $\Phi$ terms cancel. What remains depends mainly on the start state, not on the action sequence, so maximizing shaped return maximizes the same policy as the original return.

### 5. Shaped Q-learning learns useful values earlier

Now we train tabular Q-learning twice: once with the sparse reward and once with the shaped reward. The update is $Q(s,a)\leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]$. Shaping changes only the reward inside the target.

In [ ]:
def state_id_w(state_w):
    return state_w[0] * W_w + state_w[1]

def train_q_w(shaped_w, episodes_w=180, alpha_w=0.4, epsilon_w=0.2):
    Q_w = np.zeros((H_w * W_w, 4))
    lengths_w = []
    for ep_w in range(episodes_w):
        state_w = start_w
        steps_w = 0
        for _ in range(60):
            sid_w = state_id_w(state_w)
            action_w = np.random.randint(4) if np.random.rand() < epsilon_w else int(np.argmax(Q_w[sid_w]))
            nxt_w, reward_w, done_w = step_w(state_w, action_w)
            if shaped_w:
                reward_w += gamma_w * Phi_w[nxt_w] - Phi_w[state_w]
            target_w = reward_w + gamma_w * np.max(Q_w[state_id_w(nxt_w)]) * (not done_w)
            Q_w[sid_w, action_w] += alpha_w * (target_w - Q_w[sid_w, action_w])
            state_w = nxt_w
            steps_w += 1
            if done_w:
                break
        lengths_w.append(steps_w)
    return Q_w, np.array(lengths_w)

np.random.seed(2)
Q_base_w, len_base_w = train_q_w(False)
np.random.seed(2)
Q_shape_w, len_shape_w = train_q_w(True)

print("first-40 avg lengths base/shaped:", round(len_base_w[:40].mean(), 2), round(len_shape_w[:40].mean(), 2))
print("last-40 avg lengths base/shaped:", round(len_base_w[-40:].mean(), 2), round(len_shape_w[-40:].mean(), 2))

assert len_shape_w[-40:].mean() < 20

▶ What you'll see: shaped learning reaches short episodes quickly while both eventually find goal-reaching behavior.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(np.convolve(len_base_w, np.ones(10)/10, mode="valid"), label="base reward", color="gray")
plt.plot(np.convolve(len_shape_w, np.ones(10)/10, mode="valid"), label="shaped reward", color="teal")
plt.title("5: shaped Q-learning episode length")
plt.xlabel("episode"); plt.ylabel("10-episode moving average steps"); plt.legend(); plt.show()

▶ What you'll see: the shaped curve usually drops sooner because every progress-making move gives a denser target.

*Why it's done this way:* Q-learning bootstraps from its own estimates, so sparse rewards make many early targets nearly identical. Potential-based shaping changes the learning signal without changing the underlying optimal policy, giving the update more informative temporal-difference errors before the terminal reward is frequently discovered.

### 6. Bad shaping can change the task

Not every extra reward is safe. If we add an arbitrary action bonus, such as paying the agent for moving right regardless of state, the bonus is not a potential difference and may alter the optimal behavior.

In [ ]:
right_bonus_w = 2.0
loop_path_w = [(0, 0), (0, 1), (0, 0), (0, 1), (0, 0), (0, 1)]
base_loop_w = sum(-1.0 for _ in range(len(loop_path_w) - 1))
bad_loop_w = sum((-1.0 + (right_bonus_w if loop_path_w[t + 1][1] > loop_path_w[t][1] else 0.0)) for t in range(len(loop_path_w) - 1))

print("base loop return:", base_loop_w, "bad-shaped loop return:", bad_loop_w)

assert bad_loop_w > base_loop_w

▶ What you'll see: the arbitrary right bonus makes a useless loop look less bad.

In [ ]:
labels_w = ["goal path base", "goal path potential", "loop base", "loop bad bonus"]
vals_w = [sum(rewards_w), sum(rewards_w) + 6.0, base_loop_w, bad_loop_w]
plt.figure(figsize=(5, 3))
plt.bar(labels_w, vals_w, color=["gray", "teal", "gray", "crimson"])
plt.xticks(rotation=25); plt.ylabel("undiscounted demo return"); plt.title("6: unsafe shaping changes incentives")
plt.show()

▶ What you'll see: the unsafe bonus rewards behavior for the wrong reason, unlike the potential-based progress bonus.

*Why it's done this way:* Reward shaping is part of the objective design. Potential-based shaping is special because its discounted sum cancels into a start-state constant; arbitrary bonuses do not cancel, so they can create reward hacking and change what “optimal” means.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses a
> handful of small numbers, prints every intermediate value with an inline `# ->`, draws one
> picture, and ends with an `assert`. Run them top to bottom.

### ✍️ Toy 1 · Sparse rewards delay useful feedback

In a sparse gridworld, most moves pay the same step cost. The big positive signal appears only when the goal is reached.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_height = 3                                    # -> 3
t1_width = 3                                     # -> 3
t1_start = (0, 0)                                # -> (0, 0)
t1_goal = (2, 2)                                 # -> (2, 2)
t1_path = [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]  # -> shortest path
t1_rewards = np.array([-1.0, -1.0, -1.0, 10.0]) # -> [-1.0, -1.0, -1.0, 10.0]
t1_return = float(t1_rewards.sum())              # -> 7.0

print("path:", t1_path)                          # -> [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]
print("rewards:", t1_rewards.tolist())           # -> [-1.0, -1.0, -1.0, 10.0]
print("undiscounted return:", t1_return)         # -> 7.0

assert t1_return == 7.0

t1_grid = np.zeros((t1_height, t1_width))         # -> 3x3 zeros
t1_grid[t1_start] = 1.0                          # -> start marker
t1_grid[t1_goal] = 2.0                           # -> goal marker
plt.figure(figsize=(3.4, 3.0))
plt.imshow(t1_grid, cmap="viridis", vmin=0, vmax=2)
plt.xticks(range(t1_width))
plt.yticks(range(t1_height))
plt.colorbar(label="0 empty, 1 start, 2 goal")
plt.title("Toy 1 · sparse terminal payoff")
plt.show()

▶ What you'll see: three identical penalties arrive before the single terminal `+10` reward.

### ✍️ Toy 2 · A potential function scores progress toward the goal

A potential assigns a scalar to each state. Negative Manhattan distance makes values climb toward zero near the goal.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_height = 3                                    # -> 3
t2_width = 3                                     # -> 3
t2_goal = (2, 2)                                 # -> (2, 2)
t2_rows = np.arange(t2_height)[:, None]          # -> [[0], [1], [2]]
t2_cols = np.arange(t2_width)[None, :]           # -> [[0, 1, 2]]
t2_distance = np.abs(t2_rows - t2_goal[0]) + np.abs(t2_cols - t2_goal[1])  # -> Manhattan distances
t2_phi = -t2_distance.astype(float)              # -> [[-4, -3, -2], [-3, -2, -1], [-2, -1, 0]]

print("distance to goal:\n", t2_distance)        # -> [[4, 3, 2], [3, 2, 1], [2, 1, 0]]
print("potential Phi:\n", t2_phi)                # -> [[-4, -3, -2], [-3, -2, -1], [-2, -1, 0]]
print("Phi(start):", t2_phi[0, 0])               # -> -4.0
print("Phi(goal):", t2_phi[t2_goal])             # -> 0.0

assert t2_phi[0, 0] == -4.0
assert t2_phi[t2_goal] == 0.0

plt.figure(figsize=(3.6, 3.0))
plt.imshow(t2_phi, cmap="plasma")
for t2_r in range(t2_height):
    for t2_c in range(t2_width):
        plt.text(t2_c, t2_r, int(t2_phi[t2_r, t2_c]), ha="center", va="center", color="white")
plt.xticks(range(t2_width))
plt.yticks(range(t2_height))
plt.colorbar(label="Phi")
plt.title("Toy 2 · potential rises toward goal")
plt.show()

▶ What you'll see: potentials are most negative far from the goal and become 0 at the goal.

### ✍️ Toy 3 · Potential-based shaping rewards a potential increase

The shaping term `γΦ(s') − Φ(s)` is larger when a move increases potential toward the goal.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_gamma = 0.9                                   # -> 0.9
t3_base_reward = -1.0                            # -> -1.0
t3_phi_start = -4.0                              # -> -4.0
t3_phi_right = -3.0                              # -> -3.0
t3_phi_wall = -4.0                               # -> -4.0
t3_F_right = t3_gamma * t3_phi_right - t3_phi_start  # -> 1.3
t3_F_wall = t3_gamma * t3_phi_wall - t3_phi_start     # -> 0.4
t3_shaped_right = t3_base_reward + t3_F_right    # -> 0.3
t3_shaped_wall = t3_base_reward + t3_F_wall      # -> -0.6

print("F for right move:", round(t3_F_right, 3)) # -> 1.3
print("F for wall bump:", round(t3_F_wall, 3))   # -> 0.4
print("shaped right reward:", round(t3_shaped_right, 3))  # -> 0.3
print("shaped wall reward:", round(t3_shaped_wall, 3))    # -> -0.6

assert round(t3_F_right, 3) == 1.3
assert t3_shaped_right > t3_shaped_wall

plt.figure(figsize=(4.2, 2.8))
plt.bar(["right", "wall"], [t3_F_right, t3_F_wall], color=["teal", "gray"])
plt.ylabel("F(s,a,s')")
plt.title("Toy 3 · progress gets more shaping reward")
plt.show()

▶ What you'll see: moving right receives a larger shaping bonus than bumping into the wall.

### ✍️ Toy 4 · The shaping sum telescopes to a start-state constant

For potential-based shaping, the discounted shaping additions are the same for paths with the same start and terminal potential, so rankings stay fixed.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_gamma = 0.9                                   # -> 0.9
t4_goal = (2, 2)                                 # -> (2, 2)
t4_short = [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]  # -> 4 moves
t4_detour = [(0, 0), (1, 0), (0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]  # -> 6 moves

def t4_phi(t4_state):
    return -float(abs(t4_state[0] - t4_goal[0]) + abs(t4_state[1] - t4_goal[1]))

def t4_return(t4_path, t4_shaped):
    t4_total = 0.0
    for t4_i in range(len(t4_path) - 1):
        t4_base = 10.0 if t4_path[t4_i + 1] == t4_goal else -1.0
        t4_bonus = t4_gamma * t4_phi(t4_path[t4_i + 1]) - t4_phi(t4_path[t4_i]) if t4_shaped else 0.0
        t4_total = t4_total + (t4_gamma ** t4_i) * (t4_base + t4_bonus)
    return t4_total

t4_base_short = t4_return(t4_short, False)       # -> 4.58
t4_shape_short = t4_return(t4_short, True)       # -> 8.58
t4_base_detour = t4_return(t4_detour, False)     # -> 1.8098
t4_shape_detour = t4_return(t4_detour, True)     # -> 5.8098
t4_add_short = t4_shape_short - t4_base_short    # -> 4.0
t4_add_detour = t4_shape_detour - t4_base_detour # -> 4.0

print("base returns:", round(t4_base_short, 4), round(t4_base_detour, 4))      # -> 4.58 1.8098
print("shaped returns:", round(t4_shape_short, 4), round(t4_shape_detour, 4))  # -> 8.58 5.8098
print("shaping additions:", round(t4_add_short, 4), round(t4_add_detour, 4))   # -> 4.0 4.0

assert round(t4_add_short, 4) == round(t4_add_detour, 4)
assert t4_shape_short > t4_shape_detour

plt.figure(figsize=(4.8, 2.8))
plt.bar(["short base", "detour base", "short shaped", "detour shaped"], [t4_base_short, t4_base_detour, t4_shape_short, t4_shape_detour], color=["gray", "gray", "teal", "teal"])
plt.xticks(rotation=20)
plt.ylabel("discounted return")
plt.title("Toy 4 · shaping preserves the ranking")
plt.show()

▶ What you'll see: shaping adds `4.0` to both paths, so the shorter path remains better.

### ✍️ Toy 5 · Shaping changes the Q-learning target, not the update rule

A shaped reward enters the same TD target formula. Here the progress bonus makes the one-step update positive instead of negative.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_gamma = 0.9                                   # -> 0.9
t5_alpha = 0.5                                   # -> 0.5
t5_old_q = 0.0                                   # -> 0.0
t5_base_reward = -1.0                            # -> -1.0
t5_shaping_bonus = 1.3                           # -> 1.3
t5_next_value = 0.5                              # -> 0.5
t5_base_target = t5_base_reward + t5_gamma * t5_next_value  # -> -0.55
t5_shaped_reward = t5_base_reward + t5_shaping_bonus        # -> 0.3
t5_shaped_target = t5_shaped_reward + t5_gamma * t5_next_value  # -> 0.75
t5_base_new_q = t5_old_q + t5_alpha * (t5_base_target - t5_old_q)  # -> -0.275
t5_shaped_new_q = t5_old_q + t5_alpha * (t5_shaped_target - t5_old_q)  # -> 0.375

print("base target:", round(t5_base_target, 3))   # -> -0.55
print("shaped reward:", round(t5_shaped_reward, 3))  # -> 0.3
print("shaped target:", round(t5_shaped_target, 3))  # -> 0.75
print("new Q values:", round(t5_base_new_q, 3), round(t5_shaped_new_q, 3))  # -> -0.275 0.375

assert round(t5_shaped_new_q, 3) == 0.375
assert t5_shaped_new_q > t5_base_new_q

plt.figure(figsize=(4.2, 2.8))
plt.bar(["base update", "shaped update"], [t5_base_new_q, t5_shaped_new_q], color=["gray", "teal"])
plt.ylabel("new Q")
plt.title("Toy 5 · shaping gives a denser TD signal")
plt.show()

▶ What you'll see: the shaped update moves Q upward because progress turns the immediate target positive.

### ✍️ Toy 6 · Arbitrary action bonuses can reward the wrong behavior

A bonus for moving right is not a potential difference. If it is large enough, a useless right-left loop can outscore the real goal path.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_goal_return = 7.0                             # -> shortest sparse path return
t6_loop_moves = np.array(["right", "left", "right", "left"])  # -> useless loop
t6_base_loop = -1.0 * len(t6_loop_moves)         # -> -4.0
t6_right_bonus = 6.0                             # -> 6.0
t6_right_count = int(np.sum(t6_loop_moves == "right"))  # -> 2
t6_bad_loop = t6_base_loop + t6_right_bonus * t6_right_count  # -> 8.0

print("goal return:", t6_goal_return)            # -> 7.0
print("loop moves:", t6_loop_moves.tolist())     # -> ["right", "left", "right", "left"]
print("base loop return:", t6_base_loop)         # -> -4.0
print("bad-shaped loop return:", t6_bad_loop)    # -> 8.0

assert t6_bad_loop > t6_goal_return
assert t6_right_count == 2

plt.figure(figsize=(4.2, 2.8))
plt.bar(["goal path", "loop base", "loop bad bonus"], [t6_goal_return, t6_base_loop, t6_bad_loop], color=["teal", "gray", "crimson"])
plt.ylabel("undiscounted return")
plt.title("Toy 6 · unsafe shaping can flip incentives")
plt.show()

▶ What you'll see: the arbitrary right bonus makes the loop score `8.0`, beating the real goal path's `7.0`.


## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, tabular RL values, and numerical checks.
import matplotlib.pyplot as plt  # load Matplotlib for heatmaps, arrows, and learning curves.
np.random.seed(0)  # make all stochastic examples reproducible across runs.

## 🟢 Basics (warm-up)

### Basic 1 — Make a gridworld state table

**Goal.** Number every grid cell as a state, because tabular RL stores one row of values per state. We build it in 2 steps.

In [ ]:
H_b1, W_b1 = 3, 4
states_b1 = np.arange(H_b1 * W_b1).reshape(H_b1, W_b1)
goal_b1 = (2, 3)

print("state table:\n", states_b1)

▶ What you'll see: a 3×4 table whose entries are state ids.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(states_b1, cmap="viridis")
for r_b1 in range(H_b1):
    for c_b1 in range(W_b1):
        plt.text(c_b1, r_b1, states_b1[r_b1, c_b1], ha="center", va="center", color="white")
plt.title("Basic 1: state ids")
plt.colorbar(label="state id")
plt.show()

▶ What you'll see: each grid location has one integer address.

👀 Takeaway: tabular reward shaping starts by making states indexable.

### Basic 2 — Define deterministic grid moves

**Goal.** Implement boundary-safe actions, because shaping is applied to transitions from one state to the next. We build it in 2 steps.

In [ ]:
H_b2, W_b2 = 3, 4
moves_b2 = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}
state_b2 = (0, 0)

print("available moves:", moves_b2)

▶ What you'll see: four action names mapped to row-column changes.

In [ ]:
def next_state_b2(state_b2, action_b2):
    dr_b2, dc_b2 = moves_b2[action_b2]
    return (int(np.clip(state_b2[0] + dr_b2, 0, H_b2 - 1)), int(np.clip(state_b2[1] + dc_b2, 0, W_b2 - 1)))

print("from", state_b2, "right ->", next_state_b2(state_b2, "right"))
print("from", state_b2, "up ->", next_state_b2(state_b2, "up"))

assert next_state_b2(state_b2, "up") == (0, 0)

▶ What you'll see: moving up at the top boundary leaves the agent in place.

In [ ]:
plt.figure(figsize=(4, 3))
for r_b2 in range(H_b2):
    for c_b2 in range(W_b2):
        plt.scatter(c_b2, r_b2, s=220, color="lightsteelblue", edgecolor="black")
        plt.text(c_b2, r_b2, f"({r_b2},{c_b2})", ha="center", va="center", fontsize=8)

for action_b2, (dr_b2, dc_b2) in moves_b2.items():
    nr_b2, nc_b2 = next_state_b2(state_b2, action_b2)
    plt.arrow(state_b2[1], state_b2[0], nc_b2 - state_b2[1], nr_b2 - state_b2[0],
              head_width=0.08, length_includes_head=True, color="crimson")
    plt.text(nc_b2 + 0.05, nr_b2 - 0.05, action_b2, color="crimson", fontsize=8)

plt.gca().invert_yaxis()
plt.xticks(range(W_b2)); plt.yticks(range(H_b2))
plt.title("Basic 2: deterministic moves from (0, 0)")
plt.grid(True, alpha=0.3)
plt.show()

▶ What you'll see: arrows from the top-left state show valid movement right/down and clipped boundary moves up/left.

👀 Takeaway: reward shaping compares the potential of the actual next state, including boundary effects.

### Basic 3 — Compute a sparse transition reward

**Goal.** Return -1 per step and +10 at the goal, because delayed rewards create the shaping motivation. We build it in 2 steps.

In [ ]:
goal_b3 = (2, 3)
path_b3 = [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3)]

print("path:", path_b3)

▶ What you'll see: a five-step route from start to goal.

In [ ]:
rewards_b3 = np.array([10.0 if path_b3[i + 1] == goal_b3 else -1.0 for i in range(len(path_b3) - 1)])

print("transition rewards:", rewards_b3.tolist(), "sum:", rewards_b3.sum())

assert rewards_b3.sum() == 6.0
plt.figure(figsize=(4, 3))
plt.bar(range(len(rewards_b3)), rewards_b3, color="teal")
plt.title("Basic 3: sparse rewards along a path")
plt.xlabel("transition"); plt.ylabel("reward"); plt.show()

▶ What you'll see: most transitions are negative until the final goal reward.

👀 Takeaway: sparse rewards tell the agent little until it reaches the terminal state.

### Basic 4 — Discount a return

**Goal.** Weight future rewards by powers of γ, because RL values delayed consequences less than immediate ones. We build it in 2 steps.

In [ ]:
rewards_b4 = np.array([-1.0, -1.0, -1.0, 10.0])
gamma_b4 = 0.9
powers_b4 = gamma_b4 ** np.arange(len(rewards_b4))

print("discount powers:", np.round(powers_b4, 3))

▶ What you'll see: powers 1, 0.9, 0.81, 0.729.

In [ ]:
G_b4 = float(np.sum(powers_b4 * rewards_b4))

print("discounted return:", round(G_b4, 3))

assert round(G_b4, 3) == 4.58
plt.figure(figsize=(4, 3))
plt.bar(range(len(rewards_b4)), powers_b4 * rewards_b4, color="purple")
plt.title("Basic 4: discounted reward contributions")
plt.xlabel("time"); plt.ylabel("γ^t r_t"); plt.show()

▶ What you'll see: the terminal reward still dominates, but it is discounted by γ³.

👀 Takeaway: shaping must use the same γ as the return to preserve policy rankings.

### Basic 5 — Build a distance-based potential

**Goal.** Turn distance to the goal into a state potential, because potential-based shaping needs one scalar Φ(s). We build it in 2 steps.

In [ ]:
H_b5, W_b5 = 3, 4
goal_b5 = (2, 3)
Phi_b5 = np.zeros((H_b5, W_b5))
for r_b5 in range(H_b5):
    for c_b5 in range(W_b5):
        Phi_b5[r_b5, c_b5] = -(abs(r_b5 - goal_b5[0]) + abs(c_b5 - goal_b5[1]))

print("Phi:\n", Phi_b5)

assert Phi_b5[0, 0] == -5 and Phi_b5[2, 3] == 0

▶ What you'll see: potentials become less negative near the goal.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(Phi_b5, cmap="plasma")
plt.title("Basic 5: potential surface")
plt.colorbar(label="Φ(s)")
plt.show()

▶ What you'll see: a smooth gradient pointing toward the goal.

👀 Takeaway: a potential is a progress score, not the environment reward itself.

### Basic 6 — Compute one shaping bonus

**Goal.** Evaluate F = γΦ(s') − Φ(s), because this is the safe potential-based shaping term. We build it in 2 steps.

In [ ]:
Phi_b6 = np.array([[-5., -4., -3., -2.], [-4., -3., -2., -1.], [-3., -2., -1., 0.]])
gamma_b6 = 0.9
s_b6 = (0, 0)
sp_b6 = (0, 1)

print("Phi(s):", Phi_b6[s_b6], "Phi(s'):", Phi_b6[sp_b6])

▶ What you'll see: the next state has higher potential.

In [ ]:
F_b6 = gamma_b6 * Phi_b6[sp_b6] - Phi_b6[s_b6]

print("shaping bonus:", round(float(F_b6), 3))

assert round(float(F_b6), 3) == 1.4
plt.figure(figsize=(4, 3))
plt.bar(["γΦ(s')", "-Φ(s)", "F"], [gamma_b6 * Phi_b6[sp_b6], -Phi_b6[s_b6], F_b6], color="teal")
plt.title("Basic 6: pieces of F")
plt.show()

▶ What you'll see: the progress move receives a positive shaping bonus.

👀 Takeaway: potential-based shaping rewards increases in discounted potential.

### Basic 7 — Add shaping to an environment reward

**Goal.** Form R' = R + F, because learning sees the shaped reward while the task reward remains separate. We build it in 2 steps.

In [ ]:
base_reward_b7 = -1.0
F_b7 = 1.4
shaped_reward_b7 = base_reward_b7 + F_b7

print("base reward:", base_reward_b7, "bonus:", F_b7)

▶ What you'll see: the original step penalty and shaping bonus are separate numbers.

In [ ]:
print("shaped reward:", round(shaped_reward_b7, 3))

assert round(shaped_reward_b7, 3) == 0.4
plt.figure(figsize=(4, 3))
plt.bar(["base R", "bonus F", "shaped R'"], [base_reward_b7, F_b7, shaped_reward_b7], color=["gray", "teal", "purple"])
plt.title("Basic 7: reward plus shaping")
plt.show()

▶ What you'll see: a progress step can become positive even though the base task gives -1.

👀 Takeaway: shaping changes the training signal, not the definition of task success.

### Basic 8 — Compare progress and no-progress moves

**Goal.** Score two transitions from the same state, because a useful potential distinguishes progress from wasting time. We build it in 2 steps.

In [ ]:
Phi_b8 = np.array([[-5., -4., -3., -2.], [-4., -3., -2., -1.], [-3., -2., -1., 0.]])
gamma_b8 = 0.9
s_b8 = (0, 0)
nexts_b8 = {"right": (0, 1), "up-wall": (0, 0)}

print("candidate next states:", nexts_b8)

▶ What you'll see: one move progresses and one hits the wall.

In [ ]:
bonuses_b8 = np.array([gamma_b8 * Phi_b8[nexts_b8[name_b8]] - Phi_b8[s_b8] for name_b8 in nexts_b8])

print("bonuses:", dict(zip(nexts_b8.keys(), np.round(bonuses_b8, 3))))

assert np.allclose(np.round(bonuses_b8, 3), [1.4, 0.5])
plt.figure(figsize=(4, 3))
plt.bar(list(nexts_b8.keys()), bonuses_b8, color=["teal", "gray"])
plt.title("Basic 8: progress earns more shaping")
plt.ylabel("F"); plt.show()

▶ What you'll see: moving right receives the larger bonus.

👀 Takeaway: good potentials provide dense local preferences aligned with long-term progress.

### Basic 9 — Write one Q-learning target

**Goal.** Combine reward, discount, and next-state value, because shaping enters Q-learning through the target. We build it in 2 steps.

In [ ]:
r_b9 = 0.4
next_q_b9 = np.array([1.0, 1.5, 0.2, 0.7])
gamma_b9 = 0.9

print("next Q values:", next_q_b9)

▶ What you'll see: the next state has four possible action values.

In [ ]:
target_b9 = r_b9 + gamma_b9 * np.max(next_q_b9)

print("one-step target:", round(float(target_b9), 3))

assert round(float(target_b9), 3) == 1.75
plt.figure(figsize=(4, 3))
plt.bar(["r'", "γ max Q", "target"], [r_b9, gamma_b9 * np.max(next_q_b9), target_b9], color="orange")
plt.title("Basic 9: shaped TD target")
plt.show()

▶ What you'll see: the target is the shaped immediate reward plus bootstrapped future value.

👀 Takeaway: reward shaping changes the TD target by changing the immediate reward term.

### Basic 10 — Apply one Q update

**Goal.** Move one Q value toward its target, because learning is an incremental correction rather than a full overwrite. We build it in 2 steps.

In [ ]:
old_q_b10 = 0.2
target_b10 = 1.75
alpha_b10 = 0.4
td_error_b10 = target_b10 - old_q_b10

print("TD error:", round(td_error_b10, 3))

▶ What you'll see: the current value is below the target.

In [ ]:
new_q_b10 = old_q_b10 + alpha_b10 * td_error_b10

print("new Q:", round(new_q_b10, 3))

assert round(new_q_b10, 3) == 0.82
plt.figure(figsize=(4, 3))
plt.bar(["old Q", "new Q", "target"], [old_q_b10, new_q_b10, target_b10], color=["gray", "teal", "black"])
plt.title("Basic 10: one Q-learning update")
plt.show()

▶ What you'll see: the updated value moves partway toward the target.

👀 Takeaway: shaping can accelerate learning because it changes the errors that Q-learning follows.

## 🟡 Easy

### Easy 1 — Simulate a shaped path return

**Goal.** Compare base and shaped discounted returns for one path, because shaping should add guidance while keeping the same objective ordering. We build it in 3 steps.

In [ ]:
path_e1 = [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3)]
goal_e1 = (2, 3)
gamma_e1 = 0.9
Phi_e1 = np.array([[-5., -4., -3., -2.], [-4., -3., -2., -1.], [-3., -2., -1., 0.]])

print("path length:", len(path_e1) - 1)

▶ What you'll see: the path reaches the goal in five transitions.

In [ ]:
base_terms_e1 = []
shape_terms_e1 = []
for t_e1 in range(len(path_e1) - 1):
    r_e1 = 10.0 if path_e1[t_e1 + 1] == goal_e1 else -1.0
    F_e1 = gamma_e1 * Phi_e1[path_e1[t_e1 + 1]] - Phi_e1[path_e1[t_e1]]
    base_terms_e1.append((gamma_e1 ** t_e1) * r_e1)
    shape_terms_e1.append((gamma_e1 ** t_e1) * (r_e1 + F_e1))

print("base return:", round(float(np.sum(base_terms_e1)), 3))
print("shaped return:", round(float(np.sum(shape_terms_e1)), 3))

▶ What you'll see: the shaped return is higher because progress bonuses were added.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(base_terms_e1, marker="o", label="base discounted terms")
plt.plot(shape_terms_e1, marker="o", label="shaped discounted terms")
plt.title("Easy 1: return terms along one path")
plt.xlabel("transition"); plt.ylabel("discounted term"); plt.legend(); plt.show()

▶ What you'll see: shaped terms give more informative feedback before the final payoff.

👀 Takeaway: shaped rewards densify the learning signal along a trajectory.

### Easy 2 — Verify path ranking is preserved

**Goal.** Compare a short path and a detour under base and potential-shaped rewards, because policy invariance means rankings from the same start should not flip. We build it in 3 steps.

In [ ]:
gamma_e2 = 0.9
goal_e2 = (2, 3)
Phi_e2 = np.array([[-5., -4., -3., -2.], [-4., -3., -2., -1.], [-3., -2., -1., 0.]])
short_e2 = [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3)]
detour_e2 = [(0, 0), (1, 0), (0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3)]

print("lengths:", len(short_e2) - 1, len(detour_e2) - 1)

▶ What you'll see: the detour has two extra transitions.

In [ ]:
def ret_e2(path_e2, shaped_e2):
    total_e2 = 0.0
    for t_e2 in range(len(path_e2) - 1):
        r_e2 = 10.0 if path_e2[t_e2 + 1] == goal_e2 else -1.0
        if shaped_e2:
            r_e2 += gamma_e2 * Phi_e2[path_e2[t_e2 + 1]] - Phi_e2[path_e2[t_e2]]
        total_e2 += (gamma_e2 ** t_e2) * r_e2
    return float(total_e2)

returns_e2 = np.array([ret_e2(short_e2, False), ret_e2(detour_e2, False), ret_e2(short_e2, True), ret_e2(detour_e2, True)])

print("returns:", np.round(returns_e2, 3))

assert returns_e2[0] > returns_e2[1] and returns_e2[2] > returns_e2[3]

▶ What you'll see: the short path remains better with and without shaping.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["short base", "detour base", "short shaped", "detour shaped"], returns_e2, color=["gray", "gray", "teal", "teal"])
plt.xticks(rotation=25); plt.ylabel("discounted return"); plt.title("Easy 2: ranking preserved")
plt.show()

▶ What you'll see: potential shaping shifts values without reversing the preference.

👀 Takeaway: potential-based shaping preserves optimal policies because it adds a telescoping constant from a fixed start.

### Easy 3 — Train a tiny shaped Q learner

**Goal.** Run Q-learning with potential-shaped rewards, because the dense signal should produce a goal-reaching greedy policy. We build it in 4 steps.

In [ ]:
H_e3, W_e3 = 3, 4
start_e3, goal_e3 = (0, 0), (2, 3)
actions_e3 = np.array([[-1, 0], [1, 0], [0, -1], [0, 1]])
Phi_e3 = np.array([[-5., -4., -3., -2.], [-4., -3., -2., -1.], [-3., -2., -1., 0.]])

print("states:", H_e3 * W_e3, "actions:", len(actions_e3))

▶ What you'll see: a small tabular control problem.

In [ ]:
def sid_e3(s_e3):
    return s_e3[0] * W_e3 + s_e3[1]

def step_e3(s_e3, a_e3):
    ns_e3 = (int(np.clip(s_e3[0] + actions_e3[a_e3, 0], 0, H_e3 - 1)), int(np.clip(s_e3[1] + actions_e3[a_e3, 1], 0, W_e3 - 1)))
    done_e3 = ns_e3 == goal_e3
    return ns_e3, (10.0 if done_e3 else -1.0), done_e3

print("test right:", step_e3(start_e3, 3))

▶ What you'll see: the transition function returns next state, reward, and done flag.

In [ ]:
np.random.seed(3)
Q_e3 = np.zeros((H_e3 * W_e3, 4))
gamma_e3 = 0.9
for ep_e3 in range(120):
    s_e3 = start_e3
    for _ in range(40):
        a_e3 = np.random.randint(4) if np.random.rand() < 0.25 else int(np.argmax(Q_e3[sid_e3(s_e3)]))
        ns_e3, r_e3, done_e3 = step_e3(s_e3, a_e3)
        r_e3 += gamma_e3 * Phi_e3[ns_e3] - Phi_e3[s_e3]
        target_e3 = r_e3 + gamma_e3 * np.max(Q_e3[sid_e3(ns_e3)]) * (not done_e3)
        Q_e3[sid_e3(s_e3), a_e3] += 0.4 * (target_e3 - Q_e3[sid_e3(s_e3), a_e3])
        s_e3 = ns_e3
        if done_e3:
            break

print("start Q:", np.round(Q_e3[sid_e3(start_e3)], 2))

▶ What you'll see: right/down actions from the start become valuable.

In [ ]:
policy_e3 = np.argmax(Q_e3, axis=1).reshape(H_e3, W_e3)

print("greedy action ids:\n", policy_e3)

plt.figure(figsize=(4, 3))
plt.imshow(policy_e3, cmap="tab10")
plt.title("Easy 3: learned greedy action id")
plt.colorbar(label="0 up, 1 down, 2 left, 3 right")
plt.show()

▶ What you'll see: most arrows point toward the goal corridor.

👀 Takeaway: potential shaping makes early Q-learning targets more directional.

### Easy 4 — Compare base versus shaped learning curves

**Goal.** Train two learners from the same random seed, because shaping is useful only if it changes learning speed without changing the target task. We build it in 4 steps.

In [ ]:
H_e4, W_e4 = 3, 4
start_e4, goal_e4 = (0, 0), (2, 3)
actions_e4 = np.array([[-1, 0], [1, 0], [0, -1], [0, 1]])
Phi_e4 = np.array([[-5., -4., -3., -2.], [-4., -3., -2., -1.], [-3., -2., -1., 0.]])

print("experiment grid:", H_e4, "x", W_e4)

▶ What you'll see: the same grid is used for both learners.

In [ ]:
def run_e4(shaped_e4):
    Q_e4 = np.zeros((H_e4 * W_e4, 4))
    lengths_e4 = []
    for ep_e4 in range(100):
        s_e4 = start_e4
        for t_e4 in range(40):
            sid0_e4 = s_e4[0] * W_e4 + s_e4[1]
            a_e4 = np.random.randint(4) if np.random.rand() < 0.25 else int(np.argmax(Q_e4[sid0_e4]))
            ns_e4 = (int(np.clip(s_e4[0] + actions_e4[a_e4, 0], 0, H_e4 - 1)), int(np.clip(s_e4[1] + actions_e4[a_e4, 1], 0, W_e4 - 1)))
            done_e4 = ns_e4 == goal_e4
            r_e4 = 10.0 if done_e4 else -1.0
            if shaped_e4:
                r_e4 += 0.9 * Phi_e4[ns_e4] - Phi_e4[s_e4]
            sid1_e4 = ns_e4[0] * W_e4 + ns_e4[1]
            Q_e4[sid0_e4, a_e4] += 0.4 * (r_e4 + 0.9 * np.max(Q_e4[sid1_e4]) * (not done_e4) - Q_e4[sid0_e4, a_e4])
            s_e4 = ns_e4
            if done_e4:
                break
        lengths_e4.append(t_e4 + 1)
    return np.array(lengths_e4)

np.random.seed(4)
base_len_e4 = run_e4(False)
np.random.seed(4)
shape_len_e4 = run_e4(True)

print("last-20 averages:", round(base_len_e4[-20:].mean(), 2), round(shape_len_e4[-20:].mean(), 2))

▶ What you'll see: both learners improve, with shaped learning often shorter earlier.

In [ ]:
smooth_base_e4 = np.convolve(base_len_e4, np.ones(8)/8, mode="valid")
smooth_shape_e4 = np.convolve(shape_len_e4, np.ones(8)/8, mode="valid")

print("first smoothed values:", round(float(smooth_base_e4[0]), 2), round(float(smooth_shape_e4[0]), 2))

▶ What you'll see: moving averages make the noisy episode lengths easier to compare.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(smooth_base_e4, label="base", color="gray")
plt.plot(smooth_shape_e4, label="potential-shaped", color="teal")
plt.title("Easy 4: learning speed comparison")
plt.xlabel("episode"); plt.ylabel("moving-average steps"); plt.legend(); plt.show()

▶ What you'll see: shaped rewards usually reduce wandering sooner.

👀 Takeaway: reward shaping is about faster learning, not a different goal.

### Easy 5 — Visualize the learned value surface

**Goal.** Convert learned Q values into state values, because value heatmaps show whether the agent sees the goal as attractive. We build it in 3 steps.

In [ ]:
Q_e5 = Q_e3.copy()
V_e5 = np.max(Q_e5, axis=1).reshape(3, 4)

print("value surface:\n", np.round(V_e5, 2))

▶ What you'll see: states closer to the goal tend to have larger values.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(V_e5, cmap="viridis")
plt.colorbar(label="max_a Q(s,a)")
plt.title("Easy 5: value surface after shaping")
for r_e5 in range(3):
    for c_e5 in range(4):
        plt.text(c_e5, r_e5, round(float(V_e5[r_e5, c_e5]), 1), ha="center", va="center", color="white")
plt.show()

▶ What you'll see: a ramp of high values leading to the goal.

In [ ]:
best_start_e5 = int(np.argmax(Q_e5[0]))

print("best start action id:", best_start_e5)

assert best_start_e5 in [1, 3]

▶ What you'll see: the best start action is down or right, both shortest-path moves.

👀 Takeaway: a shaped learner should still prefer actions that reach the original goal efficiently.

## 🔴 Advanced

### Advanced 1 — Show telescoping numerically

**Goal.** Sum discounted shaping terms and compare them with the telescoped expression, because this is the algebra behind policy invariance. We build it in 3 steps.

In [ ]:
path_a1 = [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3)]
Phi_a1 = np.array([[-5., -4., -3., -2.], [-4., -3., -2., -1.], [-3., -2., -1., 0.]])
gamma_a1 = 0.9
terms_a1 = []

print("start potential:", Phi_a1[path_a1[0]], "end potential:", Phi_a1[path_a1[-1]])

▶ What you'll see: the trajectory starts at negative potential and ends at zero.

In [ ]:
for t_a1 in range(len(path_a1) - 1):
    F_a1 = gamma_a1 * Phi_a1[path_a1[t_a1 + 1]] - Phi_a1[path_a1[t_a1]]
    terms_a1.append((gamma_a1 ** t_a1) * F_a1)
sum_terms_a1 = float(np.sum(terms_a1))
telescope_a1 = float(-(Phi_a1[path_a1[0]]) + (gamma_a1 ** (len(path_a1) - 1)) * Phi_a1[path_a1[-1]])

print("sum discounted F:", round(sum_terms_a1, 6), "telescope:", round(telescope_a1, 6))

assert abs(sum_terms_a1 - telescope_a1) < 1e-9

▶ What you'll see: the explicit sum equals the compact telescoping formula.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(range(len(terms_a1)), terms_a1, color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Advanced 1: discounted shaping terms")
plt.xlabel("transition"); plt.ylabel("γ^t F_t"); plt.show()

▶ What you'll see: individual bonuses vary, but their discounted sum collapses to the endpoint expression.

👀 Takeaway: policy invariance comes from cancellation of intermediate potentials.

### Advanced 2 — Compare two potentials

**Goal.** Scale a potential and inspect bonuses, because potential magnitude changes learning speed and variance even when policy invariance still holds. We build it in 3 steps.

In [ ]:
Phi_base_a2 = np.array([[-5., -4., -3., -2.], [-4., -3., -2., -1.], [-3., -2., -1., 0.]])
scales_a2 = np.array([0.5, 1.0, 2.0])
gamma_a2 = 0.9
s_a2, sp_a2 = (0, 0), (0, 1)

print("scales:", scales_a2)

▶ What you'll see: the same shape of potential will be tested at three strengths.

In [ ]:
bonuses_a2 = np.array([gamma_a2 * (scale_a2 * Phi_base_a2[sp_a2]) - (scale_a2 * Phi_base_a2[s_a2]) for scale_a2 in scales_a2])

print("bonuses:", np.round(bonuses_a2, 3))

assert np.allclose(np.round(bonuses_a2, 3), [0.7, 1.4, 2.8])

▶ What you'll see: doubling the potential doubles the shaping bonus.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(scales_a2, bonuses_a2, marker="o", color="purple")
plt.title("Advanced 2: potential scale changes bonus size")
plt.xlabel("potential scale"); plt.ylabel("F for progress move"); plt.show()

▶ What you'll see: a linear relationship between potential scale and reward bonus.

👀 Takeaway: potential-based shaping is policy-safe, but the scale is still a learning-stability hyperparameter.

### Advanced 3 — Detect unsafe action bonuses

**Goal.** Compare potential shaping with an arbitrary right-action bonus, because only the potential difference has the invariance guarantee. We build it in 3 steps.

In [ ]:
path_goal_a3 = [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3)]
path_loop_a3 = [(0, 0), (0, 1), (0, 0), (0, 1), (0, 0), (0, 1)]
right_bonus_a3 = 2.0

print("right moves in loop:", sum(path_loop_a3[t + 1][1] > path_loop_a3[t][1] for t in range(len(path_loop_a3) - 1)))

▶ What you'll see: the loop repeatedly earns the arbitrary right-move bonus.

In [ ]:
def bad_return_a3(path_a3):
    total_a3 = 0.0
    for t_a3 in range(len(path_a3) - 1):
        base_a3 = 10.0 if path_a3[t_a3 + 1] == (2, 3) else -1.0
        bonus_a3 = right_bonus_a3 if path_a3[t_a3 + 1][1] > path_a3[t_a3][1] else 0.0
        total_a3 += base_a3 + bonus_a3
    return float(total_a3)

vals_a3 = np.array([bad_return_a3(path_goal_a3), bad_return_a3(path_loop_a3)])

print("bad-shaped returns goal/loop:", vals_a3)

assert vals_a3[1] > -5

▶ What you'll see: the loop is artificially improved by a bonus unrelated to task completion.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["goal path", "right-loop"], vals_a3, color=["teal", "crimson"])
plt.title("Advanced 3: arbitrary bonus changes incentives")
plt.ylabel("undiscounted return"); plt.show()

▶ What you'll see: unsafe shaping can make non-goal behavior look attractive.

👀 Takeaway: reward hacking often starts when shaping rewards are not potential differences.

### Advanced 4 — Train with too-large shaping scale

**Goal.** Compare moderate and oversized potential scales, because dense rewards can destabilize bootstrapping if their magnitude dominates the task reward. We build it in 4 steps.

In [ ]:
H_a4, W_a4 = 3, 4
start_a4, goal_a4 = (0, 0), (2, 3)
Phi_a4 = np.array([[-5., -4., -3., -2.], [-4., -3., -2., -1.], [-3., -2., -1., 0.]])
actions_a4 = np.array([[-1, 0], [1, 0], [0, -1], [0, 1]])

print("potential range:", Phi_a4.min(), Phi_a4.max())

▶ What you'll see: the potential spans five units before scaling.

In [ ]:
def train_scale_a4(scale_a4):
    np.random.seed(10)
    Q_a4 = np.zeros((H_a4 * W_a4, 4))
    maxabs_a4 = []
    for ep_a4 in range(80):
        s_a4 = start_a4
        for _ in range(35):
            sid_a4 = s_a4[0] * W_a4 + s_a4[1]
            a_a4 = np.random.randint(4) if np.random.rand() < 0.25 else int(np.argmax(Q_a4[sid_a4]))
            ns_a4 = (int(np.clip(s_a4[0] + actions_a4[a_a4, 0], 0, H_a4 - 1)), int(np.clip(s_a4[1] + actions_a4[a_a4, 1], 0, W_a4 - 1)))
            done_a4 = ns_a4 == goal_a4
            r_a4 = (10.0 if done_a4 else -1.0) + 0.9 * scale_a4 * Phi_a4[ns_a4] - scale_a4 * Phi_a4[s_a4]
            sidn_a4 = ns_a4[0] * W_a4 + ns_a4[1]
            Q_a4[sid_a4, a_a4] += 0.4 * (r_a4 + 0.9 * np.max(Q_a4[sidn_a4]) * (not done_a4) - Q_a4[sid_a4, a_a4])
            s_a4 = ns_a4
            if done_a4:
                break
        maxabs_a4.append(float(np.max(np.abs(Q_a4))))
    return np.array(maxabs_a4)

curve1_a4 = train_scale_a4(1.0)
curve8_a4 = train_scale_a4(8.0)

print("final max |Q|:", round(curve1_a4[-1], 2), round(curve8_a4[-1], 2))

▶ What you'll see: the oversized scale produces much larger value magnitudes.

In [ ]:
ratio_a4 = curve8_a4[-1] / curve1_a4[-1]

print("magnitude ratio:", round(float(ratio_a4), 2))

assert ratio_a4 > 2.0

▶ What you'll see: shaped values can become several times larger when the potential is scaled up.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(curve1_a4, label="scale 1", color="teal")
plt.plot(curve8_a4, label="scale 8", color="crimson")
plt.title("Advanced 4: shaping scale affects value magnitude")
plt.xlabel("episode"); plt.ylabel("max |Q|"); plt.legend(); plt.show()

▶ What you'll see: large shaping scales inflate TD targets and can make updates harder to tune.

👀 Takeaway: policy-safe does not mean numerically harmless; shaping scale still needs tuning.

### Advanced 5 — Evaluate greedy policy length

**Goal.** Roll out the greedy policy learned with shaping on the original environment, because final evaluation should use task rewards, not shaped rewards. We build it in 4 steps.

In [ ]:
H_a5, W_a5 = 3, 4
start_a5, goal_a5 = (0, 0), (2, 3)
actions_a5 = np.array([[-1, 0], [1, 0], [0, -1], [0, 1]])
Phi_a5 = np.array([[-5., -4., -3., -2.], [-4., -3., -2., -1.], [-3., -2., -1., 0.]])
Q_a5 = np.zeros((H_a5 * W_a5, 4))
np.random.seed(30)
for ep_a5 in range(120):
    s_train_a5 = start_a5
    for _ in range(40):
        sid_train_a5 = s_train_a5[0] * W_a5 + s_train_a5[1]
        a_train_a5 = np.random.randint(4) if np.random.rand() < 0.25 else int(np.argmax(Q_a5[sid_train_a5]))
        ns_train_a5 = (int(np.clip(s_train_a5[0] + actions_a5[a_train_a5, 0], 0, H_a5 - 1)), int(np.clip(s_train_a5[1] + actions_a5[a_train_a5, 1], 0, W_a5 - 1)))
        done_train_a5 = ns_train_a5 == goal_a5
        r_train_a5 = (10.0 if done_train_a5 else -1.0) + 0.9 * Phi_a5[ns_train_a5] - Phi_a5[s_train_a5]
        sid_next_a5 = ns_train_a5[0] * W_a5 + ns_train_a5[1]
        Q_a5[sid_train_a5, a_train_a5] += 0.4 * (r_train_a5 + 0.9 * np.max(Q_a5[sid_next_a5]) * (not done_train_a5) - Q_a5[sid_train_a5, a_train_a5])
        s_train_a5 = ns_train_a5
        if done_train_a5:
            break

print("start greedy Q:", np.round(Q_a5[0], 2))

▶ What you'll see: the learned start-state action values from the shaped learner.

In [ ]:
s_a5 = start_a5
visited_a5 = [s_a5]
base_rewards_a5 = []
for _ in range(10):
    sid_a5 = s_a5[0] * W_a5 + s_a5[1]
    a_a5 = int(np.argmax(Q_a5[sid_a5]))
    ns_a5 = (int(np.clip(s_a5[0] + actions_a5[a_a5, 0], 0, H_a5 - 1)), int(np.clip(s_a5[1] + actions_a5[a_a5, 1], 0, W_a5 - 1)))
    done_a5 = ns_a5 == goal_a5
    base_rewards_a5.append(10.0 if done_a5 else -1.0)
    visited_a5.append(ns_a5)
    s_a5 = ns_a5
    if done_a5:
        break

print("visited:", visited_a5)

▶ What you'll see: the greedy policy walks from start to goal.

In [ ]:
base_return_a5 = float(np.sum(base_rewards_a5))

print("base rewards:", base_rewards_a5, "base return:", base_return_a5)

assert visited_a5[-1] == goal_a5

▶ What you'll see: evaluation ignores shaping and reports the original task reward.

In [ ]:
path_grid_a5 = np.zeros((H_a5, W_a5))
for k_a5, cell_a5 in enumerate(visited_a5):
    path_grid_a5[cell_a5] = k_a5 + 1
plt.figure(figsize=(4, 3))
plt.imshow(path_grid_a5, cmap="viridis")
plt.title("Advanced 5: greedy rollout on base task")
plt.colorbar(label="visit order")
plt.show()

▶ What you'll see: the rollout reaches the goal in a short path under the original rewards.

👀 Takeaway: shape during training, but judge success on the unshaped environment objective.